# Notebook 1: การดึงข้อมูลจากเอกสารด้วย Docling
# (Document Extraction with Docling)

## สิ่งที่จะได้เรียนรู้ (What you'll learn)
- ติดตั้งและใช้งาน Docling สำหรับแปลงเอกสาร
- แปลงไฟล์ PDF และ PPTX ภาษาไทย
- สำรวจโครงสร้าง DoclingDocument
- ดึงตาราง รูปภาพ และลำดับชั้นของเอกสาร
- บันทึกผลลัพธ์เป็น Markdown และ JSON

In [ ]:
# ตรวจสอบการติดตั้ง (Verify installation)
import docling
print(f"Docling version: {docling.__version__}")

from docling.document_converter import DocumentConverter
print("DocumentConverter imported successfully!")

## DocumentConverter คืออะไร?

`DocumentConverter` เป็นตัวแปลงเอกสารหลักของ Docling ที่รองรับหลายรูปแบบ:
- **PDF** — รวมถึง scanned PDF ที่ต้องใช้ OCR
- **PPTX** — ไฟล์ PowerPoint
- **DOCX** — ไฟล์ Word
- **HTML, Images, และอื่นๆ**

ผลลัพธ์จะได้เป็น `DoclingDocument` ซึ่งเป็นโครงสร้างข้อมูลแบบรวม (unified representation)

In [ ]:
from docling.document_converter import DocumentConverter
from pathlib import Path

# กำหนดไฟล์ต้นทาง (Set source file)
pdf_path = Path("../data/samples/thai_sample.pdf")

# สร้าง converter และแปลงเอกสาร
converter = DocumentConverter()
result = converter.convert(str(pdf_path))

print(f"สถานะการแปลง: {result.status}")
print(f"ชื่อไฟล์: {result.input.file.name}")

In [ ]:
# ดูผลลัพธ์เป็น Markdown
markdown_text = result.document.export_to_markdown()

print("=" * 60)
print("Markdown Output (500 ตัวอักษรแรก):")
print("=" * 60)
print(markdown_text[:500])
print(f"\n... (ทั้งหมด {len(markdown_text)} ตัวอักษร)")

In [ ]:
import json

# ดูโครงสร้าง JSON
json_doc = result.document.export_to_dict()

# แสดง keys ระดับบนสุด
print("Keys ในเอกสาร:")
for key in json_doc.keys():
    print(f"  - {key}")

# บันทึกเป็นไฟล์
output_path = Path("../output/extracted")
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / f"{pdf_path.stem}.json", "w", encoding="utf-8") as f:
    json.dump(json_doc, f, ensure_ascii=False, indent=2)

print(f"\nบันทึก JSON ไปที่: {output_path / f'{pdf_path.stem}.json'}")

In [ ]:
# สำรวจโครงสร้างของเอกสาร (Document hierarchy)
doc = result.document

# นับจำนวน elements
print("สรุปโครงสร้างเอกสาร:")
print(f"  จำนวนตาราง (Tables): {len(list(doc.tables))}")
print(f"  จำนวนรูปภาพ (Pictures): {len(list(doc.pictures))}")

# แสดง text elements
print("\nเนื้อหาข้อความ (แสดง 5 รายการแรก):")
for i, (item, _level) in enumerate(doc.iterate_items()):
    if i >= 5:
        print("  ...")
        break
    text_preview = item.text[:80] if hasattr(item, 'text') else str(type(item).__name__)
    print(f"  [{i}] {text_preview}")

In [ ]:
import pandas as pd

# ดึงตารางจากเอกสาร
tables = list(doc.tables)
print(f"พบตาราง {len(tables)} ตาราง\n")

for i, table in enumerate(tables):
    print(f"--- ตาราง {i+1} ---")
    df = table.export_to_dataframe(doc=doc)
    print(df.to_markdown(index=False))

    # บันทึกเป็น CSV
    csv_path = output_path / f"{pdf_path.stem}_table_{i+1}.csv"
    df.to_csv(csv_path, index=False)
    print(f"บันทึกที่: {csv_path}\n")

In [ ]:
# แปลงไฟล์ PowerPoint
pptx_path = Path("../data/samples/thai_sample.pptx")

if pptx_path.exists():
    pptx_result = converter.convert(str(pptx_path))
    pptx_markdown = pptx_result.document.export_to_markdown()

    print("PPTX Markdown Output (500 ตัวอักษรแรก):")
    print("=" * 60)
    print(pptx_markdown[:500])

    # บันทึก
    with open(output_path / f"{pptx_path.stem}.json", "w", encoding="utf-8") as f:
        json.dump(pptx_result.document.export_to_dict(), f, ensure_ascii=False, indent=2)

    # บันทึก Markdown
    with open(output_path / f"{pptx_path.stem}.md", "w", encoding="utf-8") as f:
        f.write(pptx_markdown)

    print(f"\nบันทึกผลลัพธ์ไปที่ {output_path}/")
else:
    print(f"ไม่พบไฟล์ {pptx_path} — ข้ามขั้นตอนนี้")

In [ ]:
from pathlib import Path

# แปลงไฟล์ทั้งหมดในโฟลเดอร์ data/samples/
sample_dir = Path("../data/samples")
all_files = list(sample_dir.glob("*.pdf")) + list(sample_dir.glob("*.pptx"))

print(f"พบไฟล์ {len(all_files)} ไฟล์:")
for f in all_files:
    print(f"  - {f.name}")

# แปลงทีละไฟล์
results = []
for file_path in all_files:
    print(f"\nกำลังแปลง: {file_path.name}...")
    res = converter.convert(str(file_path))
    results.append(res)

    # บันทึก Markdown
    md_path = output_path / f"{file_path.stem}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(res.document.export_to_markdown())

    # บันทึก JSON
    json_path = output_path / f"{file_path.stem}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(res.document.export_to_dict(), f, ensure_ascii=False, indent=2)

    print(f"  สถานะ: {res.status}")
    print(f"  บันทึก: {md_path.name}, {json_path.name}")

print(f"\nแปลงเสร็จสิ้น {len(results)} ไฟล์!")

## บันทึกผลลัพธ์สำหรับ Notebook ถัดไป

เราบันทึกข้อมูลที่ดึงออกมาไว้ใน `output/extracted/`:
- ไฟล์ `.md` — Markdown สำหรับอ่านง่าย
- ไฟล์ `.json` — JSON สำหรับประมวลผลต่อ
- ไฟล์ `.csv` — ตารางที่ดึงออกมา

ใน Notebook 2 เราจะนำข้อมูลเหล่านี้มาทำความสะอาดและแบ่งเป็นชิ้นส่วน (chunks)

In [ ]:
import os

# สรุปไฟล์ที่สร้าง
print("ไฟล์ที่สร้างใน output/extracted/:")
for f in sorted(output_path.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name} ({size:,} bytes)")